# Proyecto de Análisis de Señales — Ejecución en Google Colab

Este cuaderno permite clonar el repositorio en **Google Colab** (incluso siendo un repositorio **privado**) y ejecutar en orden los dos programas independientes del proyecto:

1. **PASO 1:** `python imitation.py` (codifica los 5 secretos y genera las 5 claves CSV).
2. **PASO 2:** `python reverse.py` (recupera los 5 audios a partir de los codificados y sus claves).

Los mismos archivos `imitation.py` y `reverse.py` funcionan sin cambios tanto en local (Windows/Linux) como en Google Colab.

## 1. Clonar o ubicar el repositorio

- Si el repositorio es **privado**, se te pedirá de forma segura un **GitHub Personal Access Token (PAT)** con permiso `repo` (o puedes guardarlo en *Colab Secrets* con el nombre `GITHUB_TOKEN`).
- Si el repositorio fuera **público**, se clonará directamente sin pedir token.

In [ ]:
import os
import subprocess
from getpass import getpass
from pathlib import Path

REPO_OWNER = "JuanCan00"
REPO_NAME = "Proyecto_Analisis_Senales"

# Si ya estamos dentro del repositorio (p. ej. en local o tras una ejecución previa)
if Path("imitation.py").exists() and Path("reverse.py").exists():
    print("Repositorio detectado en el directorio actual:", Path.cwd())
elif Path(REPO_NAME).exists() and (Path(REPO_NAME) / "imitation.py").exists():
    os.chdir(REPO_NAME)
    print("Directorio cambiado a:", Path.cwd())
else:
    # 1) Intentar leer GITHUB_TOKEN desde los Secrets de Google Colab (si está configurado)
    token = None
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None

    # 2) Si no hay token en Secrets, intentar clonación pública; si falla (repo privado), pedir PAT
    public_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if token:
        auth_url = f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
        subprocess.run(["git", "clone", auth_url], check=True)
    else:
        env = os.environ.copy()
        env["GIT_TERMINAL_PROMPT"] = "0"
        res = subprocess.run(["git", "clone", public_url], env=env, capture_output=True, text=True)
        if res.returncode != 0:
            print("El repositorio es privado. Ingresa tu GitHub Personal Access Token (permiso 'repo'):")
            token = getpass("GitHub Token: ").strip()
            auth_url = f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
            subprocess.run(["git", "clone", auth_url], check=True)

    os.chdir(REPO_NAME)
    print("Repositorio clonado en:", Path.cwd())

## 2. Verificar dependencias (`numpy` y `scipy`)

Google Colab ya incluye `numpy` y `scipy` preinstalados; esta celda únicamente comprueba que estén disponibles.

In [ ]:
import numpy as np
import scipy

print(f"NumPy: {np.__version__} | SciPy: {scipy.__version__}")

## 3. PASO 1 — Ejecutar `imitation.py` (Codificación)

Genera `audios/encoded/encoded_1.wav`–`encoded_5.wav` y `keys/FFT_key_1.csv`–`FFT_key_5.csv`.

In [ ]:
!python imitation.py

## 4. PASO 2 — Ejecutar `reverse.py` (Recuperación)

Lee los audios codificados y las claves existentes (sin modificarlas) y genera `audios/recovered/recovered_1.wav`–`recovered_5.wav`.

In [ ]:
!python reverse.py

## 5. Escuchar y comparar los audios directamente en Colab

Cambia `INDICE` entre `1` y `5` para escuchar el audio secreto original, su versión codificada (*Imitation*) y su versión recuperada (*Reverse*).

In [ ]:
from IPython.display import Audio, display
from scipy.io.wavfile import read
import numpy as np

INDICE = 1  # Cambiar entre 1 y 5

fs_o, raw_o = read(f"audios/originales/secret{INDICE}.wav")
orig = raw_o.astype(np.float64) / float(2 ** 31)

fs_e, enc = read(f"audios/encoded/encoded_{INDICE}.wav")
fs_r, rec = read(f"audios/recovered/recovered_{INDICE}.wav")

print(f"Secret {INDICE} — Original:")
display(Audio(orig, rate=fs_o))

print(f"Secret {INDICE} — Codificado (Imitation):")
display(Audio(enc, rate=fs_e))

print(f"Secret {INDICE} — Recuperado (Reverse):")
display(Audio(rec, rate=fs_r))